# 10 · Does call quality predict payment? (post-call model)

The brief's central claim: Stage 1's call-quality score should feed the propensity score. Here the **post-call**
model is tested on **call-anchored rows without a payment taken on the call**, target = paid in the 30 days *after*
the call. Feature groups are added one at a time:

| Run | Features |
|---|---|
| CQ-01 | pre-call features only (FS_TRANSFER) |
| CQ-02 | + call outcome (promise to pay, non-committal, contested, transferred, actions, talk time …) |
| CQ-03 | + sentiment (caller start→end, shift, agent tone; local model) |
| CQ-04 | + QA score (synthetic truth items) |
| CQ-04n | + QA score with the noise of the real AI reviewer (each of 9 items flipped with p = 0.06) |
| CQ-05 | QA + sentiment only |

Synthetic only: the real calls have no QA score yet (Stage 1 needs approval to run on them).

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
DESIGN = "call"   # the post-call model needs call rows
call_dev, _, _ = M.load_data(design=DESIGN, world=WORLD)
dev = M.post_rows(call_dev)
SPLITS = M.cv_splits(dev, target=M.TARGET_POST)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "10"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, target=M.TARGET_POST, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from salud_pipeline.paths import PHASE2_WORLDS
truth = pd.read_csv(PHASE2_WORLDS[WORLD] / "truth" / "truth_calls.csv")
items = [c for c in truth.columns if c.startswith("qa_")]
rng = np.random.default_rng(M.SEED)
noisy = truth[items].astype(float).to_numpy(); flip = rng.random(noisy.shape) < 0.06
truth["qa_noisy"] = np.where(flip, 1 - noisy, noisy).mean(axis=1)
dev = dev.merge(truth[["contact_id", "qa_noisy"]], left_on="call_id", right_on="contact_id", how="left").drop(columns="contact_id")
OUTCOME = ["inbound", "spoke_with_agent", "collection_attempted", "promised_to_pay", "non_committal", "contested_or_refused",
           "pursued_insurance", "transferred", "action_shared_balance", "action_payment_plan", "action_financial_assistance",
           "action_statement_or_bill", "action_insurance", "n_actions", "asked_financial_assistance", "talk_seconds"]
SENT = ["caller_sentiment_overall", "caller_sentiment_end", "caller_sentiment_shift", "agent_sentiment_overall"]
PRE = M.FS_TRANSFER
groups = [("CQ-01", PRE, "pre-call features only"), ("CQ-02", PRE + OUTCOME, "+ call outcome"),
          ("CQ-03", PRE + OUTCOME + SENT, "+ sentiment"), ("CQ-04", PRE + OUTCOME + SENT + ["qa_score"], "+ QA score (truth items)"),
          ("CQ-04n", PRE + OUTCOME + SENT + ["qa_noisy"], "+ QA score with AI-reviewer noise"),
          ("CQ-05", SENT + ["qa_score"], "QA + sentiment only")]
LR = lambda: M.pipeline(M.preprocess("log_z"), M.logreg(C=0.1))
GB = lambda f: M.pipeline(M.preprocess("none"), M.lgbm(f, monotone=True, num_leaves=7, learning_rate=.03, n_estimators=300,
                                                       min_child_samples=30, subsample=.8, subsample_freq=1, colsample_bytree=.8))
for rid, feats, desc in groups:
    run(f"{rid}-LR", LR(), feats, f"Post-call, L2 logistic: {desc}")
    run(f"{rid}-GB", GB(feats), feats, f"Post-call, monotone LightGBM: {desc}")

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
lb = M.leaderboard(notebook=NB)
lb["step"] = lb.run_id.str.extract(r"(CQ-\d+n?)")[0]; lb["model"] = lb.run_id.str[-2:]
piv = lb.pivot(index="step", columns="model", values="auc").loc[[g[0] for g in groups]]
piv.plot.bar(figsize=(9, 3.5), rot=0, ylim=(.5, max(.75, piv.max().max() + .02)), color=["#C44E52", "#4C72B0"])
plt.ylabel("AUC (paid after the call)"); plt.title("Adding call-feature groups one at a time"); plt.show()
tests = []
for m in ["LR", "GB"]:
    for a, b in [("CQ-02", "CQ-01"), ("CQ-03", "CQ-02"), ("CQ-04", "CQ-03"), ("CQ-04n", "CQ-03")]:
        t = M.corrected_ttest(f"{a}-{m}", f"{b}-{m}"); tests.append({"model": m, "step": f"{a} vs {b}", "AUC gain": t["mean_diff"], "p": t["p_value"]})
pd.DataFrame(tests).round(4)

## Which call features carry the signal? (L2 logistic, CQ-04n, standardised coefficients)

In [ ]:
feats = PRE + OUTCOME + SENT + ["qa_noisy"]
m = LR().fit(*M.xy(dev, feats, M.TARGET_POST))
coef = pd.Series(m[-1].coef_[0], index=m[:-1].get_feature_names_out()).sort_values()
plt.figure(figsize=(7, 0.25 * len(coef) + 1)); coef.plot.barh(color=np.where(coef > 0, "#55A868", "#C44E52"))
plt.title("Post-call model coefficients (positive = more likely to pay after the call)"); plt.show()

## Findings (Salud-tuned world)
- **Call outcome adds real signal after the call:** pre-call 0.683 rises to 0.740 with call outcome (LR, +0.057, p < 0.001); GB rises from 0.708 to 0.748 (+0.040, p < 0.001).
- **Sentiment and the QA score add nothing on top** (gain ≤ 0.001, p ≥ 0.2), even with error-free QA items. QA and sentiment alone reach only 0.57–0.58.
- **Caveat:** in this world, call quality acts through the call outcome (promise to pay, payment plan), so this mostly confirms the generator's design.
  Whether QA predicts payment in Salud's calls needs the Stage 1 QA scores for the 84 real calls, which is still waiting for approval.